In [1]:
#Import the necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import xgboost as xgb

In [2]:
df = pd.read_csv("BI_noB1B10_2017_test_pixel.csv")
df.head()

,ID,Region,tillClass,B10_p0_sow,B10_p100_sow,B10_p25_sow,B10_p50_sow,B10_p75_sow,B11_p0_sow,B11_p100_sow,...,SNDVI_p0_peak,SNDVI_p100_peak,SNDVI_p25_peak,SNDVI_p50_peak,SNDVI_p75_peak,STI_p0_peak,STI_p100_peak,STI_p25_peak,STI_p50_peak,STI_p75_peak
0,1,Bihar,ZT,-0.924913,3.029043,-0.723106,-0.209337,0.380373,0.124338,0.285878,...,0.109090,0.492909,0.238458,0.342152,0.433814,1.160622,1.510029,1.253969,1.298421,1.487457
1,1,Bihar,ZT,-1.041067,0.380373,-0.790673,-0.597369,0.137131,0.143189,0.301672,...,0.132488,0.603815,0.282442,0.409026,0.505459,1.213439,1.755575,1.366333,1.507493,1.680208
2,1,Bihar,ZT,-0.924913,3.029043,-0.723106,-0.209337,0.380373,0.131779,0.286525,...,0.110585,0.460611,0.258122,0.323402,0.395251,1.171866,1.500081,1.267745,1.335346,1.486752
3,1,Bihar,ZT,-0.924913,11.459722,-0.723106,-0.018267,0.427072,0.133647,0.407353,...,0.105139,0.458611,0.245265,0.315298,0.388134,1.155592,1.484443,1.241905,1.283196,1.411449
4,1,Bihar,ZT,-0.924913,3.029043,-0.723106,-0.209337,0.380373,0.124338,0.285878,...,0.099935,0.459526,0.248090,0.324698,0.403208,1.160622,1.510029,1.253969,1.298421,1.487457


In [3]:
from sklearn.preprocessing import LabelEncoder

# Create the encoder
le = LabelEncoder()

# Fit and transform the target column
y = le.fit_transform(df['tillClass'])

print(y)

[1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]


In [4]:
X = df.drop(['Region'], axis='columns')
selected_features = ['NDVI_p50_sow','SNDVI_p50_sow','NDTI_p50_sow','CRC_p50_sow','GCVI_p50_sow','STI_p50_sow','B10_p50_sow','B11_p50_sow']
X = df[selected_features]

In [5]:
#Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
#Split train further to train/validation
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=42)

In [6]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)  

In [7]:
xgb_classifier = xgb.XGBClassifier(random_state = 42)

xgb_classifier.fit(X_train, y_train)    #fit this model into x and y data

y_pred = xgb_classifier.predict(X_val)  #need to see how well this performs onto validation set

test_accuracy = accuracy_score(y_val, y_pred)       #yval = actual class, ypred = model predicted class
print(f"Test Accuracy : {test_accuracy:.4f}")

# Print classification report
print("\nClassification Report:\n", classification_report(y_val, y_pred))

# Confusion Matrix
print("\nConfusion Matrix:\n", confusion_matrix(y_val, y_pred))

Test Accuracy : 0.9531

Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.89      0.94        28
           1       0.92      1.00      0.96        36

    accuracy                           0.95        64
   macro avg       0.96      0.95      0.95        64
weighted avg       0.96      0.95      0.95        64


Confusion Matrix:
 [[25  3]
 [ 0 36]]


In [10]:
#Use hyperparameter training. Set parameters manually for higher accuracy.

param_grid = {  #hyper parameters below
    'n_estimators': [100, 200, 300, 500], #number of trees
    'learning_rate': [0.01, 0.1, 0.2],  #rate at which model changes
    'max_depth': [3, 6, 9], #how deep tree can get
    'min_child_weight': [1, 3, 5],
    'subsample': [0.7, 0.85, 1.0],  #controls no. of observations used to make each tree
    'colsample_bytree': [0.7, 0.85, 1.0],   #how many features used per tree. Smaller -> less complex
    'reg_alpha': [0, 0.01, 0.1, 1, 10, 100],
    'reg_lambda': [0.5, 0.7, 1, 1.3]    #don't set too low
}

xgb_model = xgb.XGBClassifier(random_state = 42)
#cross validation. 10 cross validation folds (short of cv)

#gridsearchCV will take long : every combination of parameters and loops through all of them
#RandomizedSearchCV : Gives faster process

grid_search = RandomizedSearchCV(xgb_model, param_grid, cv=10, scoring="accuracy", n_iter=100, n_jobs=-1, verbose=2, random_state=42) #n_iter : how many combos to search
grid_search.fit(X_train, y_train)

best_xgb = grid_search.best_estimator_      #return with best parameters. gives higher accuracy score

# Best parameters from tuning
print("Best Parameters:", grid_search.best_params_)
print("Best Accuracy:", grid_search.best_score_)


Fitting 10 folds for each of 100 candidates, totalling 1000 fits
Best Parameters: {'subsample': 1.0, 'reg_lambda': 1.3, 'reg_alpha': 0, 'n_estimators': 300, 'min_child_weight': 1, 'max_depth': 3, 'learning_rate': 0.1, 'colsample_bytree': 0.7}
Best Accuracy: 0.931842105263158


In [11]:
#Testing Now to X_Test
y_pred = best_xgb.predict(X_test)

test_accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {test_accuracy:.4f}")

# Print classification report
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# Confusion Matrix
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Test Accuracy: 0.8750

Classification Report:
               precision    recall  f1-score   support

           0       0.83      0.94      0.88        31
           1       0.93      0.82      0.87        33

    accuracy                           0.88        64
   macro avg       0.88      0.88      0.87        64
weighted avg       0.88      0.88      0.87        64


Confusion Matrix:
 [[29  2]
 [ 6 27]]
